# Módulo 5 · Clase 6 — Documentación y buenas prácticas

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | OpenAPI y Swagger — documentación automática y enriquecida |
| 2 | CORS — permitir llamadas desde frontends |
| 3 | Middleware — logging, tiempo de respuesta |
| 4 | Rate limiting — protección contra abuso |
| 5 | Versionado de APIs |
| 6 | Mejores prácticas de diseño REST |

In [ ]:
import subprocess, sys

for pkg in ['fastapi', 'uvicorn[standard]', 'requests', 'slowapi']:
    try:
        __import__(pkg.split('[')[0])
        print(f"{pkg.split('[')[0]}: ya instalado")
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", pkg], check=True)
        print(f"{pkg}: instalado")

---
## 1. OpenAPI y Swagger

FastAPI genera automáticamente la especificación OpenAPI 3.0. Puedes enriquecerla con metadatos:

In [ ]:
from fastapi import FastAPI, Query, Path, Body
from fastapi.openapi.utils import get_openapi
from pydantic import BaseModel, Field
from typing import Optional
import uvicorn, threading, time, requests

# Metadatos de la app
app = FastAPI(
    title="API de Proyectos v1",
    description="""
## API para gestión de proyectos de equipo

Permite crear y gestionar proyectos, tareas y miembros del equipo.

### Autenticación
Usa Bearer tokens JWT. Obtén el token en `/token`.

### Límites
- 100 requests/minuto por IP
- Payload máximo: 1MB
""",
    version="1.0.0",
    contact={"name": "Equipo API", "email": "api@empresa.com"},
    license_info={"name": "MIT"},
    openapi_tags=[
        {"name": "Proyectos", "description": "Gestión de proyectos"},
        {"name": "Tareas",    "description": "Tareas dentro de proyectos"},
        {"name": "Salud",     "description": "Estado del sistema"},
    ]
)


class ProyectoCreate(BaseModel):
    nombre:      str           = Field(
        min_length=1, max_length=100,
        description="Nombre único del proyecto",
        examples=["Web App v2", "API Migration"]
    )
    descripcion: Optional[str] = Field(None, description="Descripción detallada")
    presupuesto: Optional[float] = Field(None, gt=0, description="En euros")

    model_config = {
        "json_schema_extra": {
            "examples": [{
                "nombre": "Rediseño portal",
                "descripcion": "Nueva versión del portal de clientes",
                "presupuesto": 15000.0
            }]
        }
    }

class ProyectoResponse(ProyectoCreate):
    id: int


proyectos_db: dict[int, dict] = {}
seq = 0

@app.get("/salud", tags=["Salud"], summary="Estado del servicio")
def healthcheck():
    """Devuelve el estado actual del servicio y su versión."""
    return {"status": "ok", "version": "1.0.0"}


@app.get(
    "/proyectos",
    tags=["Proyectos"],
    summary="Listar proyectos",
    description="Devuelve todos los proyectos con soporte de paginación.",
    response_model=list[ProyectoResponse],
    responses={
        200: {"description": "Lista de proyectos"},
        401: {"description": "No autenticado"},
    }
)
def listar(
    skip:  int = Query(0,  ge=0,   description="Elementos a saltar"),
    limit: int = Query(20, ge=1, le=100, description="Máximo de resultados")
):
    items = list(proyectos_db.values())
    return items[skip:skip+limit]


@app.post(
    "/proyectos",
    tags=["Proyectos"],
    summary="Crear proyecto",
    response_model=ProyectoResponse,
    status_code=201,
    responses={
        201: {"description": "Proyecto creado"},
        409: {"description": "Nombre duplicado"},
        422: {"description": "Datos inválidos"},
    }
)
def crear(datos: ProyectoCreate):
    global seq
    if any(p['nombre'] == datos.nombre for p in proyectos_db.values()):
        from fastapi import HTTPException
        raise HTTPException(409, "Ya existe un proyecto con ese nombre")
    seq += 1
    p = {"id": seq, **datos.model_dump()}
    proyectos_db[seq] = p
    return p


cfg = uvicorn.Config(app, host="127.0.0.1", port=8030, log_level="warning")
threading.Thread(target=uvicorn.Server(cfg).run, daemon=True).start()
time.sleep(1.2)
B = "http://127.0.0.1:8030"

print("Spec OpenAPI disponible en:")
print(f"  {B}/openapi.json")
print(f"  {B}/docs        (Swagger UI)")
print(f"  {B}/redoc       (ReDoc)")

spec = requests.get(f"{B}/openapi.json").json()
print(f"\nVersión spec: {spec['openapi']}")
print(f"Título:       {spec['info']['title']}")
print(f"Endpoints:    {list(spec['paths'].keys())}")

---
## 2. CORS

```
CORS (Cross-Origin Resource Sharing):

Sin CORS:
  Frontend en https://miapp.com → API en https://api.miapp.com → ❌ BLOQUEADO

Con CORS configurado:
  Frontend en https://miapp.com → API → ✓ cabeceras CORS → ✓ OK

El navegador envía:
  OPTIONS /endpoint  Origin: https://miapp.com

La API responde:
  Access-Control-Allow-Origin: https://miapp.com
  Access-Control-Allow-Methods: GET, POST, PUT, DELETE
  Access-Control-Allow-Headers: Authorization, Content-Type
```

In [ ]:
from fastapi.middleware.cors import CORSMiddleware

app_cors = FastAPI(title="API con CORS")

# CORS — controla qué orígenes pueden llamar a la API
app_cors.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:3000",       # frontend dev
        "https://miapp.com",           # producción
        "https://admin.miapp.com",     # panel admin
    ],
    allow_credentials=True,            # permite cookies y auth headers
    allow_methods=["GET", "POST", "PUT", "PATCH", "DELETE", "OPTIONS"],
    allow_headers=["Authorization", "Content-Type", "X-Request-ID"],
    expose_headers=["X-Total-Count"],  # headers que el JS puede leer
    max_age=600,                       # tiempo en caché del preflight (segundos)
)

@app_cors.get("/datos")
def datos():
    return {"mensaje": "Con CORS configurado"}

print("CORS configurado:")
print("  - http://localhost:3000 (dev frontend)")
print("  - https://miapp.com (producción)")
print("  - https://admin.miapp.com (admin)")
print("\nPara APIs públicas usa: allow_origins=['*']")
print("⚠️  '*' no funciona con allow_credentials=True")

---
## 3. Middleware

In [ ]:
from fastapi import FastAPI, Request, Response
from starlette.middleware.base import BaseHTTPMiddleware
import time, uuid, logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("api")

app_mid = FastAPI(title="API con Middleware")


# ── Middleware 1: Logging de requests ────────────────────────
class LoggingMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        request_id = str(uuid.uuid4())[:8]
        inicio      = time.time()

        # Antes del endpoint
        logger.info(f"[{request_id}] → {request.method} {request.url.path}")

        response = await call_next(request)

        # Después del endpoint
        duracion = (time.time() - inicio) * 1000
        logger.info(f"[{request_id}] ← {response.status_code} ({duracion:.1f}ms)")

        # Añadir headers de diagnóstico
        response.headers["X-Request-ID"]      = request_id
        response.headers["X-Response-Time"]   = f"{duracion:.1f}ms"
        return response


# ── Middleware 2: Seguridad (security headers) ────────────────
class SecurityHeadersMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        response = await call_next(request)
        response.headers["X-Content-Type-Options"]   = "nosniff"
        response.headers["X-Frame-Options"]           = "DENY"
        response.headers["X-XSS-Protection"]          = "1; mode=block"
        response.headers["Strict-Transport-Security"] = "max-age=31536000"
        return response


app_mid.add_middleware(LoggingMiddleware)
app_mid.add_middleware(SecurityHeadersMiddleware)

@app_mid.get("/rapido")
def rapido(): return {"msg": "respuesta inmediata"}

@app_mid.get("/lento")
async def lento():
    import asyncio
    await asyncio.sleep(0.2)  # simular operación lenta
    return {"msg": "respuesta tras 200ms"}


cfg = uvicorn.Config(app_mid, host="127.0.0.1", port=8031, log_level="warning")
threading.Thread(target=uvicorn.Server(cfg).run, daemon=True).start()
time.sleep(1.2)
B_MID = "http://127.0.0.1:8031"

r = requests.get(f"{B_MID}/rapido")
print("Headers de diagnóstico:")
for h in ["X-Request-ID", "X-Response-Time", "X-Content-Type-Options"]:
    print(f"  {h}: {r.headers.get(h, 'no presente')}")

r2 = requests.get(f"{B_MID}/lento")
print(f"\nEndpoint lento: {r2.headers.get('X-Response-Time')}")

---
## 4. Rate Limiting

In [ ]:
from slowapi import Limiter, _rate_limit_exceeded_handler
from slowapi.util import get_remote_address
from slowapi.errors import RateLimitExceeded
from fastapi import FastAPI, Request

limiter = Limiter(key_func=get_remote_address)  # identifica por IP
app_rl  = FastAPI(title="API con Rate Limiting")
app_rl.state.limiter = limiter
app_rl.add_exception_handler(RateLimitExceeded, _rate_limit_exceeded_handler)

@app_rl.get("/datos")
@limiter.limit("5/minute")       # 5 peticiones por minuto por IP
async def datos(request: Request):
    return {"datos": [1, 2, 3]}

@app_rl.get("/pesado")
@limiter.limit("2/minute")       # endpoints costosos → límite más estricto
async def pesado(request: Request):
    return {"resultado": "operación costosa"}

@app_rl.get("/publico")
async def publico():              # sin límite
    return {"info": "acceso libre"}


cfg = uvicorn.Config(app_rl, host="127.0.0.1", port=8032, log_level="warning")
threading.Thread(target=uvicorn.Server(cfg).run, daemon=True).start()
time.sleep(1.2)
B_RL = "http://127.0.0.1:8032"

print("Probando rate limit (5/min):")
for i in range(7):
    r = requests.get(f"{B_RL}/datos")
    estado = "✓" if r.status_code == 200 else f"✗ {r.status_code}"
    print(f"  Request {i+1}: {estado}")

---
## 5. Versionado de APIs

```
Estrategias de versionado:

1. URL path (más común):
   GET /v1/usuarios
   GET /v2/usuarios

2. Header:
   Accept: application/vnd.miapi.v2+json

3. Query param:
   GET /usuarios?version=2

Cuándo versionar:
✓ Cambio que rompe la compatibilidad (breaking change)
✓ Cambio en la estructura de la respuesta
✓ Eliminación de campos

No es necesario versionar:
- Añadir campos opcionales (no rompe clientes existentes)
- Nuevos endpoints
- Corrección de bugs
```

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

# ── Routers por versión ──────────────────────────────────────
from fastapi import APIRouter

v1 = APIRouter(prefix="/v1", tags=["v1"])
v2 = APIRouter(prefix="/v2", tags=["v2"])

# V1: respuesta simple
class UsuarioV1(BaseModel):
    id:     int
    nombre: str
    email:  str

# V2: añade avatar, metadata, cambia 'nombre' → 'nombre_completo'
class UsuarioV2(BaseModel):
    id:             int
    nombre_completo:str
    email:          str
    avatar_url:     str | None = None
    metadata:       dict       = {}

USUARIOS = [
    {"id": 1, "nombre": "Ana García",   "email": "ana@x.com"},
    {"id": 2, "nombre": "Carlos López", "email": "carlos@x.com"},
]

@v1.get("/usuarios", response_model=list[UsuarioV1])
def usuarios_v1():
    return USUARIOS

@v2.get("/usuarios", response_model=list[UsuarioV2])
def usuarios_v2():
    # Transformar al nuevo formato
    return [
        {"id": u["id"], "nombre_completo": u["nombre"], "email": u["email"],
         "avatar_url": None, "metadata": {"migrado_de": "v1"}}
        for u in USUARIOS
    ]

@v1.get("/usuarios/{id}", response_model=UsuarioV1)
def usuario_v1(id: int):
    u = next((u for u in USUARIOS if u["id"] == id), None)
    if not u: from fastapi import HTTPException; raise HTTPException(404, "No encontrado")
    return u

@v2.get("/usuarios/{id}", response_model=UsuarioV2)
def usuario_v2(id: int):
    u = next((u for u in USUARIOS if u["id"] == id), None)
    if not u: from fastapi import HTTPException; raise HTTPException(404, "No encontrado")
    return {"id": u["id"], "nombre_completo": u["nombre"], "email": u["email"],
            "avatar_url": f"https://avatars.example.com/{u['id']}.png", "metadata": {}}


app_ver = FastAPI(title="API versionada")
app_ver.include_router(v1)
app_ver.include_router(v2)

# Deprecation header en v1
@app_ver.middleware("http")
async def deprecation_header(request: Request, call_next):
    response = await call_next(request)
    if request.url.path.startswith("/v1"):
        response.headers["Deprecation"]  = "true"
        response.headers["Sunset"]       = "2026-12-31T00:00:00Z"
        response.headers["Link"]         = '</v2/usuarios>; rel="successor-version"'
    return response


cfg = uvicorn.Config(app_ver, host="127.0.0.1", port=8033, log_level="warning")
threading.Thread(target=uvicorn.Server(cfg).run, daemon=True).start()
time.sleep(1.2)
B_VER = "http://127.0.0.1:8033"

print("V1 (deprecated):")
r1 = requests.get(f"{B_VER}/v1/usuarios")
print(f"  {r1.json()}")
print(f"  Deprecation: {r1.headers.get('Deprecation', 'no')}")
print(f"  Sunset:      {r1.headers.get('Sunset', 'no')}")

print("\nV2 (actual):")
r2 = requests.get(f"{B_VER}/v2/usuarios")
print(f"  {r2.json()}")

---
## 6. Mejores prácticas de diseño REST

```
✓ NOMBRES DE RECURSOS
  ✓  /usuarios          (plural, sustantivo)
  ✓  /usuarios/{id}/pedidos
  ✗  /getUsuarios       (verbos, no)
  ✗  /usuario/{id}/getPedidos

✓ VERBOS HTTP ↔ OPERACIONES
  GET    /proyectos       → listar
  POST   /proyectos       → crear
  GET    /proyectos/42    → obtener uno
  PUT    /proyectos/42    → reemplazar completo
  PATCH  /proyectos/42    → actualizar parcial
  DELETE /proyectos/42    → eliminar

✓ STATUS CODES
  200 OK             → GET, PUT, PATCH exitoso
  201 Created        → POST exitoso
  204 No Content     → DELETE exitoso
  400 Bad Request    → datos malformados
  401 Unauthorized   → no autenticado
  403 Forbidden      → autenticado pero sin permisos
  404 Not Found      → recurso no existe
  409 Conflict       → conflicto de negocio (duplicado, estado incorrecto)
  422 Unprocessable  → datos válidos pero fallan reglas (FastAPI usa esto para validación)
  429 Too Many Requests → rate limit
  500 Internal Error → error del servidor

✓ FORMATO DE ERRORES CONSISTENTE
  {
    "error":   "NOMBRE_ERROR",
    "message": "Descripción legible",
    "details": {...}
  }

✓ PAGINACIÓN
  GET /items?skip=0&limit=20
  Respuesta: {"total": 100, "skip": 0, "limit": 20, "items": [...]}

✓ IDEMPOTENCIA
  GET, PUT, DELETE → misma llamada = mismo resultado
  POST → puede crear duplicados si se llama varias veces
  PATCH → generalmente idempotente si aplica los mismos cambios
```

In [ ]:
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from fastapi.exceptions import RequestValidationError
from pydantic import BaseModel
from typing import Optional, Any

app_bp = FastAPI(title="API con buenas prácticas")

# ── Formato de error consistente ─────────────────────────────
class ErrorResponse(BaseModel):
    error:   str
    message: str
    details: Optional[Any] = None

# Interceptar errores de validación Pydantic (422)
@app_bp.exception_handler(RequestValidationError)
async def validation_handler(request: Request, exc: RequestValidationError):
    errores = [
        {"campo": ".".join(str(x) for x in e["loc"][1:]), "mensaje": e["msg"]}
        for e in exc.errors()
    ]
    return JSONResponse(
        status_code=422,
        content={"error": "VALIDATION_ERROR", "message": "Datos inválidos", "details": errores}
    )

# Interceptar HTTPException
@app_bp.exception_handler(HTTPException)
async def http_exception_handler(request: Request, exc: HTTPException):
    return JSONResponse(
        status_code=exc.status_code,
        content={"error": f"HTTP_{exc.status_code}", "message": exc.detail, "details": None}
    )


from pydantic import Field

class ItemCreate(BaseModel):
    nombre: str = Field(min_length=1)
    precio: float = Field(gt=0)

items_db: dict[int, dict] = {}
seq2 = 0

@app_bp.get("/items")
def listar_items(skip: int = 0, limit: int = 20):
    todos  = list(items_db.values())
    return {"total": len(todos), "skip": skip, "limit": limit, "items": todos[skip:skip+limit]}

@app_bp.post("/items", status_code=201)
def crear_item(datos: ItemCreate):
    global seq2; seq2 += 1
    item = {"id": seq2, **datos.model_dump()}
    items_db[seq2] = item
    return item

@app_bp.get("/items/{id}")
def get_item(id: int):
    if id not in items_db:
        raise HTTPException(status_code=404, detail=f"Item {id} no encontrado")
    return items_db[id]


cfg = uvicorn.Config(app_bp, host="127.0.0.1", port=8034, log_level="warning")
threading.Thread(target=uvicorn.Server(cfg).run, daemon=True).start()
time.sleep(1.2)
B_BP = "http://127.0.0.1:8034"

print("Error de validación (422):")
r = requests.post(f"{B_BP}/items", json={"nombre": "", "precio": -5})
import json
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

print("\nNo encontrado (404):")
r = requests.get(f"{B_BP}/items/999")
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

---
## Práctica

### Ejercicio 1 — Documentación enriquecida

**Enunciado:** Toma una API sencilla de gestión de eventos (puedes crearla desde cero o partir de la de tareas de la Clase 3) y añade documentación completa:

- Metadatos de la app (título, descripción con Markdown, versión, contacto)
- Tags para agrupar endpoints
- `summary` y `description` en cada endpoint
- `examples` en los modelos Pydantic
- `responses` documentando los status codes posibles en cada endpoint

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
from fastapi import FastAPI, HTTPException, Query
from pydantic import BaseModel, Field
from typing import Optional

app_ev = FastAPI(
    title="API de Eventos",
    description="""
## Gestión de eventos y conferencias

Crea, consulta y gestiona eventos con control de aforo.

### Autenticación
Bearer token JWT. Obtén el token en `/token`.
""",
    version="1.0.0",
    contact={"name": "Soporte", "email": "soporte@eventos.com"},
    openapi_tags=[
        {"name": "Eventos",    "description": "CRUD de eventos"},
        {"name": "Asistentes", "description": "Gestión de inscripciones"},
    ]
)

class EventoCreate(BaseModel):
    nombre:   str           = Field(min_length=1, max_length=200,
                                    examples=["PyDay Galicia 2025"])
    lugar:    str           = Field(min_length=1, examples=["Santiago de Compostela"])
    aforo:    int           = Field(gt=0, description="Plazas máximas")
    fecha:    str           = Field(description="Formato YYYY-MM-DD", examples=["2025-10-15"])
    online:   bool          = False

    model_config = {
        "json_schema_extra": {
            "examples": [{
                "nombre": "PyDay Galicia 2025",
                "lugar":  "Auditorio USC",
                "aforo":  200,
                "fecha":  "2025-10-15",
                "online": False
            }]
        }
    }

class EventoResponse(EventoCreate):
    id:         int
    inscritos:  int = 0

ev_db: dict[int, dict] = {}
ev_seq = 0

@app_ev.get(
    "/eventos",
    tags=["Eventos"],
    summary="Listar todos los eventos",
    description="Devuelve los eventos disponibles con soporte de paginación y filtros.",
    response_model=list[EventoResponse],
    responses={
        200: {"description": "Lista de eventos"},
    }
)
def listar_eventos(
    online: Optional[bool] = Query(None, description="Filtrar por formato online/presencial"),
    skip:   int            = 0,
    limit:  int            = Query(20, le=100)
):
    items = list(ev_db.values())
    if online is not None: items = [e for e in items if e['online'] == online]
    return items[skip:skip+limit]

@app_ev.post(
    "/eventos",
    tags=["Eventos"],
    summary="Crear evento",
    response_model=EventoResponse,
    status_code=201,
    responses={
        201: {"description": "Evento creado"},
        409: {"description": "Nombre de evento duplicado"},
        422: {"description": "Datos inválidos"},
    }
)
def crear_evento(datos: EventoCreate):
    global ev_seq
    ev_seq += 1
    ev = {"id": ev_seq, "inscritos": 0, **datos.model_dump()}
    ev_db[ev_seq] = ev
    return ev
```
</details>

---
### Ejercicio 2 — Middleware de métricas

**Enunciado:** Implementa un middleware que registre métricas de uso:

- Contador de requests por endpoint y método
- Tiempo de respuesta acumulado por endpoint
- `GET /metricas` — devuelve las estadísticas recopiladas
- `POST /metricas/reset` — reinicia los contadores

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
from fastapi import FastAPI, Request
from starlette.middleware.base import BaseHTTPMiddleware
import time
from collections import defaultdict

metricas: dict[str, dict] = defaultdict(lambda: {"requests": 0, "tiempo_total_ms": 0.0})

class MetricasMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        inicio   = time.time()
        response = await call_next(request)
        duracion = (time.time() - inicio) * 1000

        clave = f"{request.method} {request.url.path}"
        metricas[clave]["requests"]       += 1
        metricas[clave]["tiempo_total_ms"] += duracion

        return response

app_m = FastAPI()
app_m.add_middleware(MetricasMiddleware)

@app_m.get("/datos")  def datos(): return {"ok": True}
@app_m.get("/otros") def otros(): return {"ok": True}

@app_m.get("/metricas")
def ver_metricas():
    return {
        endpoint: {
            "requests": v["requests"],
            "tiempo_medio_ms": round(v["tiempo_total_ms"] / v["requests"], 1) if v["requests"] else 0
        }
        for endpoint, v in metricas.items()
        if not endpoint.endswith("/metricas")  # excluir el propio endpoint
    }

@app_m.post("/metricas/reset")
def reset_metricas():
    metricas.clear()
    return {"mensaje": "Métricas reiniciadas"}
```
</details>

---
### Ejercicio 3 — CORS y headers de seguridad

**Enunciado:** Configura CORS para dos entornos:

- En desarrollo (`ENV=dev`): permite todos los orígenes
- En producción (`ENV=prod`): solo permite `https://miapp.com` y `https://admin.miapp.com`
- Añade un middleware que inyecte headers de seguridad estándar
- Endpoint `GET /config` que muestre el entorno activo y los orígenes permitidos

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import os
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from starlette.middleware.base import BaseHTTPMiddleware

ENV = os.getenv("ENV", "dev")  # por defecto dev

ORIGENES = {
    "dev":  ["*"],
    "prod": ["https://miapp.com", "https://admin.miapp.com"],
}

origenes_activos = ORIGENES.get(ENV, ["*"])

app_env = FastAPI()

app_env.add_middleware(
    CORSMiddleware,
    allow_origins=origenes_activos,
    allow_credentials=ENV == "prod",  # solo en prod con orígenes específicos
    allow_methods=["*"],
    allow_headers=["Authorization", "Content-Type"],
)

SECURITY_HEADERS = {
    "X-Content-Type-Options":    "nosniff",
    "X-Frame-Options":           "DENY",
    "X-XSS-Protection":          "1; mode=block",
    "Referrer-Policy":           "strict-origin-when-cross-origin",
    "Permissions-Policy":        "camera=(), microphone=(), geolocation=()",
}

class SecurityHeaders(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        response = await call_next(request)
        for k, v in SECURITY_HEADERS.items():
            response.headers[k] = v
        if ENV == "prod":
            response.headers["Strict-Transport-Security"] = "max-age=31536000; includeSubDomains"
        return response

app_env.add_middleware(SecurityHeaders)

@app_env.get("/config")
def config():
    return {
        "entorno": ENV,
        "origenes_cors": origenes_activos,
        "headers_seguridad": list(SECURITY_HEADERS.keys()),
    }
```
</details>

---
### Ejercicio 4 — Rate limiting por usuario

**Enunciado:** Implementa rate limiting diferenciado sin slowapi:

- Usuarios autenticados: 100 requests/minuto
- Usuarios anónimos: 10 requests/minuto
- Admins: sin límite
- Implementa como middleware que inspeccione el header Authorization

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from starlette.middleware.base import BaseHTTPMiddleware
import time
from collections import defaultdict

# {clave: [(timestamp, )]}
ventana_requests: dict[str, list] = defaultdict(list)

LIMITES = {"anonimo": 10, "usuario": 100, "admin": None}  # None = sin límite

def obtener_rol(token: str | None) -> str:
    if not token: return "anonimo"
    try:
        from jose import jwt
        payload = jwt.decode(token, CLAVE_SECRETA, algorithms=["HS256"])
        return payload.get("rol", "usuario")
    except: return "anonimo"

class RateLimitMiddleware(BaseHTTPMiddleware):
    VENTANA = 60  # segundos

    async def dispatch(self, request: Request, call_next):
        auth    = request.headers.get("Authorization", "")
        token   = auth.replace("Bearer ", "") if auth.startswith("Bearer ") else None
        rol     = obtener_rol(token)
        limite  = LIMITES.get(rol)

        if limite is not None:  # admin → sin comprobación
            ip    = request.client.host
            clave = f"{ip}:{rol}"
            ahora = time.time()

            # Limpiar requests fuera de la ventana
            ventana_requests[clave] = [
                t for t in ventana_requests[clave] if ahora - t < self.VENTANA
            ]

            if len(ventana_requests[clave]) >= limite:
                return JSONResponse(
                    status_code=429,
                    content={"error": "RATE_LIMIT", "message": f"Límite: {limite} req/min para {rol}"},
                    headers={"Retry-After": "60"}
                )
            ventana_requests[clave].append(ahora)

        return await call_next(request)
```
</details>

---
### Ejercicio 5 (reto) — API de producción lista

**Enunciado:** Combina todo lo aprendido en esta clase para hacer una API de nivel producción:

Toma la API de tareas de la Clase 3 y añádele:
1. Documentación OpenAPI completa (metadatos, tags, examples, responses)
2. CORS configurado (dev vs. prod según variable de entorno)
3. Middleware de logging con `X-Request-ID` y tiempo de respuesta
4. Rate limiting: 20 req/min por IP
5. Manejo de errores con formato consistente `{error, message, details}`
6. Versionado: la API actual es `/v1/tareas`, añade `/v2/tareas` donde la respuesta incluye `tiempo_restante_dias` calculado desde `fecha_limite`

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

Este ejercicio es largo — aquí está la estructura clave:

```python
# 1. App con metadatos
app_prod = FastAPI(
    title="API Tareas v1",
    description="## API de gestión de tareas\n\nSoporta CRUD completo con autenticación.",
    version="1.0.0",
    openapi_tags=[{"name": "Tareas", "description": "Gestión de tareas del equipo"}]
)

# 2. CORS
import os
app_prod.add_middleware(CORSMiddleware,
    allow_origins=["*"] if os.getenv("ENV","dev") == "dev" else ["https://miapp.com"],
    allow_methods=["*"], allow_headers=["*"]
)

# 3. Logging middleware (ver demo de la clase)
app_prod.add_middleware(LoggingMiddleware)

# 4. Rate limiting con slowapi (ver demo)

# 5. Exception handlers
@app_prod.exception_handler(RequestValidationError)
async def val_err(request, exc):
    return JSONResponse(422, {"error": "VALIDATION_ERROR", "message": "Datos inválidos",
                               "details": exc.errors()})

# 6. Routers v1 y v2
v1 = APIRouter(prefix="/v1", tags=["Tareas"])
v2 = APIRouter(prefix="/v2", tags=["Tareas v2"])

@v1.get("/tareas") def listar_v1(): return list(bd.values())

@v2.get("/tareas")
def listar_v2():
    from datetime import date
    resultado = []
    for t in bd.values():
        t2 = dict(t)
        if t.get("fecha_limite"):
            try:
                fl = date.fromisoformat(t["fecha_limite"])
                t2["tiempo_restante_dias"] = (fl - date.today()).days
            except: t2["tiempo_restante_dias"] = None
        resultado.append(t2)
    return resultado

app_prod.include_router(v1)
app_prod.include_router(v2)
```
</details>

---
## Resumen

```python
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from starlette.middleware.base import BaseHTTPMiddleware

app = FastAPI(
    title="Mi API",
    description="Descripción con **Markdown**",
    version="1.0.0",
    openapi_tags=[{"name": "Recursos", "description": "..."}]
)

# CORS
app.add_middleware(CORSMiddleware,
    allow_origins=["https://miapp.com"],
    allow_methods=["*"], allow_headers=["Authorization", "Content-Type"]
)

# Middleware propio
class MiMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request, call_next):
        response = await call_next(request)
        response.headers["X-Custom"] = "valor"
        return response
app.add_middleware(MiMiddleware)

# Endpoint documentado
@app.get("/recursos", tags=["Recursos"], summary="Lista recursos",
         responses={200: {...}, 401: {...}})
def listar(): ...

# Versionado con routers
v1 = APIRouter(prefix="/v1")
v2 = APIRouter(prefix="/v2")
app.include_router(v1)
app.include_router(v2)
```

## ¿Qué viene en la próxima clase?

En la **Clase 7** construimos el proyecto final del módulo: una API completa con autenticación, base de datos, documentación y todas las buenas prácticas aprendidas.